# Day 35: Observability, Drift and Optimisation
**JSON logs, OpenTelemetry GenAI spans, token and cost per trace, alerts, drift detection, optimisation and scheduled evals**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai tiktoken scikit-learn numpy pandas scipy matplotlib opentelemetry-sdk

# ===== Option B: VS Code / local =====
# pip install openai tiktoken scikit-learn numpy pandas scipy matplotlib opentelemetry-sdk
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

## 2. Instrumenting requests: JSON logs and simple spans

In [ ]:
import json, time, uuid, random, re, sys
from contextlib import contextmanager
import numpy as np, pandas as pd, matplotlib.pyplot as plt

LOG_FILE = "requests.jsonl"
open(LOG_FILE, "w").close()

def mask(text):
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[email]", text)
    return re.sub(r"\+?\d[\d -]{8,}\d", "[phone]", text)

def log(**fields):
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps({"ts": time.time(), **fields}) + "\n")

@contextmanager
def span(name, trace):
    t = time.perf_counter()
    yield
    trace.append({"span": name, "ms": round((time.perf_counter() - t) * 1000, 1)})

def handle(question, release="2.3.0"):
    rid, trace = uuid.uuid4().hex[:8], []
    with span("retrieve", trace):
        time.sleep(random.uniform(0.005, 0.015)); docs = ["c7", "c2"]
    with span("generate", trace):
        time.sleep(random.uniform(0.04, 0.08))
        answer = llm(f"Context: Roaming in Japan costs 980 JPY per day.\nQuestion: {question}")
    with span("guardrail", trace):
        time.sleep(0.002); verdict = "pass"
    log(event="answer", request_id=rid, release=release, prompt_id="answer_v8", question=mask(question),
        retrieved=docs, guardrail=verdict, **usage(), latency_ms=round(sum(s["ms"] for s in trace), 1), trace=trace)
    return rid, trace

random.seed(34)
for q in ["How much is roaming in Japan?", "Email me at mei@example.com about roaming", "Call +81 90 1234 5678 about roaming"]:
    rid, trace = handle(q)
    print(rid, " | ".join(f"{s['span']} {s['ms']} ms" for s in trace))
print("\nA logged line (personal data masked):\n", open(LOG_FILE).readlines()[2])

## 3. NEW (v4): OpenTelemetry GenAI spans, and token and cost per trace
We use the OpenTelemetry SDK with an in-memory exporter, so nothing leaves the notebook. Attribute names follow the OpenTelemetry GenAI semantic conventions as described in the course sources; **check the current spec version** before you build dashboards on them. `app.*` attributes are our own, not part of the standard.

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
# ===== Option B: send to a real backend instead =====
# from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter
# provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))
tracer = provider.get_tracer("smartdesk")

def chat_span(prompt, model=MODEL):
    with tracer.start_as_current_span(f"chat {model}") as s:
        s.set_attribute("gen_ai.operation.name", "chat")
        s.set_attribute("gen_ai.request.model", model)
        s.set_attribute("gen_ai.request.temperature", 0.2)
        answer = llm(prompt, model=model)
        u = usage()
        s.set_attribute("gen_ai.response.model", model)
        s.set_attribute("gen_ai.usage.input_tokens", u["input_tokens"])
        s.set_attribute("gen_ai.usage.output_tokens", u["output_tokens"])
        return answer

def agent_request(question):
    with tracer.start_as_current_span("invoke_agent smartdesk") as root:
        root.set_attribute("gen_ai.operation.name", "invoke_agent")
        root.set_attribute("app.release", "2.3.0")
        with tracer.start_as_current_span("retrieve policies") as r:
            r.set_attribute("app.retrieved_ids", ["c7", "c2"])
        plan = chat_span("Plan the steps to answer: " + question + " Context: " + "Roaming policy text. " * 120)
        with tracer.start_as_current_span("execute_tool lookup_ticket") as t:
            t.set_attribute("gen_ai.operation.name", "execute_tool")
            t.set_attribute("gen_ai.tool.name", "lookup_ticket")
        return chat_span("Write the final answer for: " + question + " Notes: " + plan)

for q in ["Why was I charged roaming in Osaka?", "Is 5G available on my plan?", "Cancel my add-on please"]:
    agent_request(q)

spans = exporter.get_finished_spans()
print(len(spans), "spans exported")
for s in spans[:5]:
    print(f"{s.name:<32} trace {format(s.context.trace_id, '032x')[:6]} | {dict(s.attributes)}")

In [ ]:
PRICE_PER_1M = {"input": 1.00, "output": 4.00}          # PLACEHOLDER prices, not real

rows = []
for s in spans:
    a = s.attributes
    rows.append({"trace": format(s.context.trace_id, "032x")[:6], "span": s.name,
                 "input": a.get("gen_ai.usage.input_tokens", 0), "output": a.get("gen_ai.usage.output_tokens", 0),
                 "ms": (s.end_time - s.start_time) / 1e6})
df_spans = pd.DataFrame(rows)
df_spans["cost"] = (df_spans.input * PRICE_PER_1M["input"] + df_spans.output * PRICE_PER_1M["output"]) / 1e6
per_trace = df_spans.groupby("trace").agg(spans=("span", "size"), input=("input", "sum"), output=("output", "sum"),
                                          cost=("cost", "sum"), ms=("ms", "sum"))
print(per_trace.round(5).to_string())
top = df_spans.sort_values("cost", ascending=False).iloc[0]
print(f"\nMost expensive span: {top.span} in trace {top.trace} ({top.cost:.5f}, placeholder units)")
print("Alert rule idea: cost per trace above 3x the median ->", (per_trace.cost > 3 * per_trace.cost.median()).any())

## 4. A simulated month of production traffic
Planted events: a new release on 18 Dec, a latency incident on 24 Dec, and a new question topic (5G plans) growing in the last week.

In [ ]:
rng = np.random.default_rng(34)
days = pd.date_range("2026-12-01", "2026-12-30")
rows = []
for d in days:
    n = int(rng.normal(900, 80))
    release = "1.9.0" if d >= pd.Timestamp("2026-12-18") else "1.8.0"
    new_share = 0.0 if d < pd.Timestamp("2026-12-23") else min(0.25, 0.03 * (d.day - 22))
    topics = rng.choice(["billing", "network", "roaming", "5g_plan"], n, p=[0.4, 0.3, 0.3 - new_share, new_share])
    tok_out = rng.normal(260 if release == "1.9.0" else 160, 30, n)
    lat = rng.gamma(4, 450, n) + tok_out * 4 + (2500 if d == pd.Timestamp("2026-12-24") else 0)
    err = rng.random(n) < (0.06 if d == pd.Timestamp("2026-12-24") else 0.008)
    idk = np.where(topics == "5g_plan", rng.random(n) < 0.6, rng.random(n) < 0.05)
    down_p = np.where(idk, 0.5, 0.18 if release == "1.9.0" else 0.05)
    rows.append(pd.DataFrame({"day": d, "release": release, "topic": topics, "tokens_out": tok_out.round(),
                              "latency_ms": lat.round(), "error": err, "idk": idk, "thumbs_down": rng.random(n) < down_p}))
traffic = pd.concat(rows, ignore_index=True)
traffic["cost"] = (900 * PRICE_PER_1M["input"] + traffic.tokens_out * PRICE_PER_1M["output"]) / 1e6   # placeholder prices
print(len(traffic), "requests over", traffic.day.nunique(), "days")

daily = traffic.groupby("day").agg(requests=("error", "size"), p95_ms=("latency_ms", lambda s: s.quantile(0.95)),
                                   error_rate=("error", "mean"), cost=("cost", "sum"), thumbs_down=("thumbs_down", "mean"),
                                   tokens_out=("tokens_out", "mean"), idk_rate=("idk", "mean"))
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col, color in zip(axes.flat, ["requests", "p95_ms", "error_rate", "cost", "thumbs_down", "tokens_out"],
                          ["#0FA3A3", "#0FA3A3", "#FF8A3D", "#0B5D63", "#FF8A3D", "#0B5D63"]):
    daily[col].plot(ax=ax, color=color, marker=".", title=col); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Alert rules: sustained and relative

In [ ]:
baseline = daily.rolling(7, min_periods=5).median().shift(1)          # trailing 7-day baseline
rules = {
    "error_spike":           daily.error_rate > 0.05,
    "slow_fixed_4s (noisy)": daily.p95_ms > 4000,
    "slow_vs_baseline":      daily.p95_ms > baseline.p95_ms * 1.3,
    "quality_drop":          daily.thumbs_down > baseline.thumbs_down * 2,
    "idk_rising":            (daily.idk_rate > baseline.idk_rate * 1.5).rolling(2).sum() == 2,
}
for name, fired in rules.items():
    hits = [d.strftime("%m-%d") for d in daily.index[fired.fillna(False).astype(bool)]]
    print(f"{name:<22} fired on {len(hits):>2} days: {hits[:8]}{' ...' if len(hits) > 8 else ''}")

by_release = traffic.groupby("release").agg(tokens_out=("tokens_out", "mean"), p95_ms=("latency_ms", lambda s: s.quantile(0.95)),
                                            thumbs_down=("thumbs_down", "mean")).round(3)
print("\nRoot cause view, by release:\n", by_release)

### Feedback loop: thumbs-down to review queue to golden set

In [ ]:
traffic["week"] = ((traffic.day - traffic.day.min()).dt.days // 7 + 1).clip(upper=4)
queue = traffic[traffic.thumbs_down & (traffic.week == 4)].sample(frac=1, random_state=1)
print("Review queue size (week 4):", len(queue))
reviewed = queue.head(5).assign(label=["missing_doc", "missing_doc", "too_long", "missing_doc", "too_long"])
golden_new = [{"id": f"fb-{i + 1:03d}", "question": f"<real {r.topic} question from log>", "must_include": ["<expected phrase>"],
               "source": "user feedback", "failure": r.label, "owner": "mei", "added": "2026-12-14"}
              for i, r in enumerate(reviewed.itertuples())]
with open("golden_additions.jsonl", "w") as f:
    for g in golden_new:
        f.write(json.dumps(g) + "\n")
print(pd.Series([g["failure"] for g in golden_new]).value_counts().to_string())

## 5. Drift: reference and current periods with planted drift

In [ ]:
from scipy.stats import ks_2samp
rng = np.random.default_rng(35)
TEMPLATES = {
    "billing": ["Why is my bill higher this month", "I was charged twice for my plan", "How do I get a refund for roaming"],
    "network": ["No signal at home since Monday", "Mobile data is very slow downtown", "Calls drop on the train"],
    "roaming": ["How much is roaming in Korea", "Do I need to switch on roaming", "Why was I charged roaming in Osaka"],
    "5g_plan": ["Is 5G available on my plan", "How do I upgrade to the 5G Unlimited plan", "Why does my phone not show 5G"],
}
def make_week(week, n=600):
    new = 0.0 if week < 4 else 0.25
    ja = 0.10 if week < 4 else 0.25
    topics = rng.choice(list(TEMPLATES), n, p=[0.4 - new / 3, 0.3 - new / 3, 0.3 - new / 3, new])
    rows = []
    for t in topics:
        q = rng.choice(TEMPLATES[t])
        if week == 4 and rng.random() < 0.5:
            q += " and what should I do if I am travelling abroad that week"           # longer questions in week 4
        rows.append({"week": week, "topic": t, "lang": "ja" if rng.random() < ja else "en", "question": q,
                     "q_len": len(q.split()), "answer_tokens": rng.normal(170 if week < 4 else 240, 30)})
    return pd.DataFrame(rows)
data = pd.concat([make_week(w) for w in [1, 2, 3, 4]], ignore_index=True)

def psi(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-6
    c = np.histogram(cur, edges)[0] / len(cur) + 1e-6
    return float(np.sum((c - r) * np.log(c / r)))

def psi_cat(ref, cur):
    cats = sorted(set(ref) | set(cur))
    r = np.array([(ref == k).mean() for k in cats]) + 1e-3          # smoothing: a brand new category still gives a large PSI
    c = np.array([(cur == k).mean() for k in cats]) + 1e-3
    return float(np.sum((c - r) * np.log(c / r)))

w1, w4 = data[data.week == 1], data[data.week == 4]
half = len(w1) // 2
print(f"PSI question length  week1 vs week4: {psi(w1.q_len, w4.q_len):.2f}")
print(f"PSI question length  week1 half A vs B (sanity check): {psi(w1.q_len[:half], w1.q_len[half:]):.2f}")
print(f"PSI topic            week1 vs week4: {psi_cat(w1.topic, w4.topic):.2f}")
print(f"PSI language         week1 vs week4: {psi_cat(w1.lang, w4.lang):.2f}")
ks = ks_2samp(w1.answer_tokens, w4.answer_tokens)
print(f"KS test answer tokens: statistic {ks.statistic:.2f}, p-value {ks.pvalue:.1e}  (small p-value = distributions differ)")

In [ ]:
def centroid(texts):
    v = embed(list(texts)).mean(axis=0)
    return v / np.linalg.norm(v)

ref = centroid(w1.question.sample(200, random_state=1))
drift = {w: round(1 - float(ref @ centroid(data[data.week == w].question.sample(200, random_state=2))), 3) for w in [1, 2, 3, 4]}
print("Centroid drift vs week 1:", drift)
pd.Series(drift).plot(marker="o", color="#FF8A3D", title="Question embedding drift vs week 1")
plt.xlabel("week"); plt.ylabel("1 - cosine"); plt.show()

## 6. NEW (v4): Provider and model drift
The configured model name never changes, but on 9 Dec the provider starts serving a new snapshot behind it. Logging `gen_ai.response.model` on every span makes this visible. Snapshots and scores here are simulated.

In [ ]:
rng = np.random.default_rng(9)
pdays = pd.date_range("2026-12-01", "2026-12-16")
prows = []
for d in pdays:
    snap = "snapshot-b" if d >= pd.Timestamp("2026-12-09") else "snapshot-a"
    prows.append({"day": d, "requested_model": MODEL, "response_model": f"{MODEL}-{snap}",
                  "tokens_out": rng.normal(230 if snap == "snapshot-b" else 170, 8),
                  "golden_pass": rng.normal(0.85 if snap == "snapshot-b" else 0.95, 0.01)})
plog = pd.DataFrame(prows)

changes = plog[plog.response_model != plog.response_model.shift()].iloc[1:]
for r in changes.itertuples():
    print(f"response model changed on {r.day:%Y-%m-%d}: requested {r.requested_model} -> served {r.response_model}")
summary = plog.groupby("response_model").agg(tokens_out=("tokens_out", "mean"), golden_pass=("golden_pass", "mean")).round(2)
print(summary)
before, after = summary.iloc[0], summary.iloc[1]
print(f"output tokens {after.tokens_out / before.tokens_out - 1:+.0%} | golden pass rate {before.golden_pass:.2f} -> {after.golden_pass:.2f}")
print("Action:", "run full evals, adjust the prompt or roll back; follow the Day 33 runbook"
      if after.golden_pass < 0.9 else "no action")

## 7. Concept drift: the golden set must change too

In [ ]:
golden_old = {"q": "What is the refund period?", "must_include": ["30 days"]}
golden_new = {"q": "What is the refund period?", "must_include": ["14 days"], "must_not_include": ["30 days"],
              "version": 2, "changed": "policy update 2026-12-01"}
def passes(g, answer):
    return all(m in answer for m in g["must_include"]) and not any(m in answer for m in g.get("must_not_include", []))
stale_answer = "Refunds are accepted within 30 days."
print("Old golden item, stale answer: ", "PASSES (wrong!)" if passes(golden_old, stale_answer) else "fails")
print("New golden item, stale answer: ", "passes" if passes(golden_new, stale_answer) else "FAILS (correct)")

## 8. Optimisation experiments: change one thing at a time
Compare: baseline, k=5, a simple keyword re-ranker on the top 10, adding the missing 5G document, and the data fix plus re-ranking.

In [ ]:
KNOWLEDGE = """Bills are issued on the 1st of each month. Duplicate charges are refunded within 14 days.
Roaming in Korea costs 980 JPY per day on the Travel plan. Roaming must be switched on in the app before travel.
Roaming charges in Japan do not apply because Japan is the home network.
No signal at home can be reported in the app; an engineer visits within 3 working days.
Slow mobile data downtown is usually caused by congestion between 18:00 and 21:00.
Calls that drop on trains are a known coverage gap; a fix is planned for March."""
FIVE_G_DOC = """5G is available on the 5G Unlimited and 5G Lite plans only.
Upgrades to 5G Unlimited are done in the app and take effect the next day.
Phones show 5G only when 5G is switched on in network settings and the plan includes 5G."""

def build_index(text):
    chunks = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    return chunks, embed(chunks)

GOLDEN = [("How long do duplicate charge refunds take?", "within 14 days"), ("How much is roaming in Korea?", "980 JPY"),
          ("When is the engineer visit for no signal?", "3 working days"), ("Why is data slow downtown in the evening?", "congestion"),
          ("Is 5G available on my plan?", "5G Lite"), ("How do I upgrade to 5G Unlimited?", "in the app"),
          ("Why does my phone not show 5G?", "network settings")]

def rerank(q, chunks, idx):
    qw = set(re.findall(r"[a-z0-9]+", q.lower()))
    return sorted(idx, key=lambda i: -len(qw & set(re.findall(r"[a-z0-9]+", chunks[i].lower()))))

def hit_rate(chunks, vecs, k, use_rerank=False):
    hits = 0
    for q, frag in GOLDEN:
        cand = list(np.argsort(vecs @ embed(q)[0])[::-1][:10 if use_rerank else k])
        top = rerank(q, chunks, cand)[:k] if use_rerank else cand
        hits += any(frag in chunks[i] for i in top)
    return hits / len(GOLDEN)

base = build_index(KNOWLEDGE)
fixed = build_index(KNOWLEDGE + "\n" + FIVE_G_DOC)
results = {"baseline (k=3)": hit_rate(*base, 3), "k=5": hit_rate(*base, 5), "re-rank top 10 (k=3)": hit_rate(*base, 3, True),
           "add 5G document (k=3)": hit_rate(*fixed, 3), "5G document + re-rank": hit_rate(*fixed, 3, True)}
for name, r in results.items():
    print(f"{name:<24} hit rate {r:.2f}")
print("\nThe data fix (adding the missing document) beats tuning. Re-ranking cannot help when the evidence is missing.")

## 9. NEW (v4): Scheduled evals with CI or Codex automations
A weekly job computes drift and golden results, writes a report and exits non-zero on a breach. A CI schedule or a Codex automation runs it; a person reviews every breach.

In [ ]:
%%writefile weekly_job.py
import sys
import numpy as np, pandas as pd

THRESHOLDS = {"psi_topic": 0.25, "psi_length": 0.25, "golden_hit_rate": 0.9}

def psi_cat(ref, cur):
    cats = sorted(set(ref) | set(cur))
    r = np.array([(ref == k).mean() for k in cats]) + 1e-3
    c = np.array([(cur == k).mean() for k in cats]) + 1e-3
    return float(np.sum((c - r) * np.log(c / r)))

def psi_num(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-3
    c = np.histogram(cur, edges)[0] / len(cur) + 1e-3
    return float(np.sum((c - r) * np.log(c / r)))

def main(ref_path, cur_path, golden_hit_rate, week):
    ref, cur = pd.read_csv(ref_path), pd.read_csv(cur_path)
    m = {"psi_topic": round(psi_cat(ref.topic, cur.topic), 2),
         "psi_length": round(psi_num(ref.q_len, cur.q_len), 2),
         "golden_hit_rate": golden_hit_rate}
    breaches = [k for k, v in m.items() if (v > THRESHOLDS[k] if k.startswith("psi") else v < THRESHOLDS[k])]
    lines = [f"# LLMOps weekly report {week}", "", "| Metric | Value | Threshold | Status |", "|---|---|---|---|"]
    lines += [f"| {k} | {v} | {THRESHOLDS[k]} | {'REVIEW' if k in breaches else 'ok'} |" for k, v in m.items()]
    lines += ["", "Action: " + ("review the breached metrics this week." if breaches else "no action needed.")]
    open(f"report_{week}.md", "w").write("\n".join(lines))
    print("\n".join(lines))
    return 1 if breaches else 0

if __name__ == "__main__":
    sys.exit(main(sys.argv[1], sys.argv[2], float(sys.argv[3]), sys.argv[4]))

In [ ]:
import subprocess
w1.to_csv("week1.csv", index=False); w4.to_csv("week4.csv", index=False)
r = subprocess.run([sys.executable, "weekly_job.py", "week1.csv", "week4.csv", str(round(results["baseline (k=3)"], 2)), "2026-W51"],
                   capture_output=True, text=True)
print(r.stdout)
print("exit code:", r.returncode, "(1 means a person should review; the scheduler opens a ticket or alert)")

In [ ]:
%%writefile azure-pipelines-weekly.yml
# Weekly drift and eval job. The key comes from a secret pipeline variable; the job also runs offline.
schedules:
- cron: "0 3 * * 1"
  displayName: Weekly drift and eval
  branches:
    include: [main]
  always: true
pool:
  vmImage: ubuntu-latest
steps:
- script: pip install numpy pandas scipy
  displayName: Install
- script: python weekly_job.py data/ref.csv data/cur.csv 0.95 $(Build.BuildNumber)
  displayName: Drift and golden set
  env:
    OPENAI_API_KEY: $(OPENAI_API_KEY)
- publish: report_$(Build.BuildNumber).md
  artifact: weekly-report
  condition: always()

In [ ]:
CODEX_AUTOMATION_PROMPT = """Every Monday at 09:00 IST:
1. Pull main. Run: python weekly_job.py data/ref.csv data/cur.csv <golden_hit_rate> <ISO week>.
2. If the exit code is 0, post a two-line summary of the report.
3. If the exit code is 1, open a draft issue titled 'Weekly drift review <week>' with the report attached
   and the three most likely causes. Do not change prompts, code or data. A person decides the fix.
Never print secrets. Use the approved model only."""
print(CODEX_AUTOMATION_PROMPT)

**TODO (SmartDesk step 30):** add the KS test on answer tokens to `weekly_job.py` with a p-value threshold of 0.01, add a `cost_per_trace_p95` metric from section 3, and write the capstone hand-off notes in your README.

In [ ]:
# TODO: write your code here

---
**Otsukaresama deshita (お疲れ様でした): *Thank you for your hard work today***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)